# BANELO SALES FORECASTING - ENHANCED ML MODEL COMPARISON
## Comprehensive ML Algorithm Evaluation

This notebook compares **6 different machine learning algorithms** for sales forecasting:
1. **Linear Regression** (Baseline Model)
2. **Decision Tree Regressor**
3. **Random Forest Regressor** (Bagging Ensemble)
4. **Gradient Boosting Regressor** (Boosting Ensemble)
5. **XGBoost Regressor** (Advanced Gradient Boosting)
6. **LightGBM Regressor** (Fast Gradient Boosting)

Models are evaluated using MAE, RMSE, MAPE, and R² metrics.

**Additional Features:**
- Top-selling products prediction by category (Beverage & Pastry)
- CSV export capability
- PDF report generation

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✅ All dependencies loaded!')

## Upload Sales Data

Click the folder icon on the left, then upload your `sales_for_colab.csv` file

In [ ]:
from google.colab import files

print('📁 Upload your sales_for_colab.csv file...')
uploaded = files.upload()

if 'sales_for_colab.csv' in uploaded:
    df = pd.read_csv('sales_for_colab.csv')
    print('✅ File uploaded successfully!')
else:
    print('❌ Expected file not found')

## Data Exploration

In [ ]:
print('=' * 70)
print('DATA OVERVIEW')
print('=' * 70)

print(f'\n📊 Dataset Shape: {df.shape[0]} records')
print(f'\n📅 Date Range: {df["date"].min()} to {df["date"].max()}')
print(f'\n🏷️  Categories: {df["category"].unique().tolist()}')
print(f'📦 Products: {df["product_name"].nunique()}')
print(f'\n📝 Sample Data:')
print(df.head(10))

## Data Preparation

In [ ]:
df['date'] = pd.to_datetime(df['date'])

daily_sales = df.groupby(['date', 'product_name', 'category']).agg({
    'quantity': 'sum',
    'total': 'sum',
    'price': 'mean'
}).reset_index().sort_values('date')

print(f'✅ Created {len(daily_sales)} daily records')
print(f'   Unique products: {daily_sales["product_name"].nunique()}')

## Feature Engineering

In [ ]:
print('\n' + '=' * 70)
print('FEATURE ENGINEERING')
print('=' * 70)

df_features = daily_sales.copy()

# TIME-BASED FEATURES
print('\n⏰ Creating time-based features...')
df_features['day_of_week'] = df_features['date'].dt.dayofweek
df_features['day_of_month'] = df_features['date'].dt.day
df_features['month'] = df_features['date'].dt.month
df_features['year'] = df_features['date'].dt.year
df_features['week_of_year'] = df_features['date'].dt.isocalendar().week
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)
df_features['days_since_start'] = (df_features['date'] - df_features['date'].min()).dt.days

# CATEGORY ENCODING
print('\n🏷️  Encoding categorical features...')
category_encoder = LabelEncoder()
df_features['category_encoded'] = category_encoder.fit_transform(df_features['category'])

# LAG FEATURES
print('\n📉 Creating lag features...')
for lag in [1, 7, 14]:
    df_features[f'quantity_lag_{lag}'] = df_features.groupby('product_name')['quantity'].shift(lag)

# ROLLING STATISTICS
print('\n📊 Creating rolling statistics...')
for window in [7, 14, 30]:
    df_features[f'quantity_ma_{window}'] = df_features.groupby('product_name')['quantity'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

df_features = df_features.fillna(0)

feature_columns = [
    'day_of_week', 'day_of_month', 'month', 'year', 'week_of_year',
    'is_weekend', 'days_since_start', 'category_encoded',
    'quantity_lag_1', 'quantity_lag_7', 'quantity_lag_14',
    'quantity_ma_7', 'quantity_ma_14', 'quantity_ma_30'
]

print(f'\n✅ Total features: {len(feature_columns)}')

## Split Data for Training

In [ ]:
print('\n' + '=' * 70)
print('DATA SPLIT')
print('=' * 70)

X = df_features[feature_columns]
y = df_features['quantity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'\n📊 Data Split:')
print(f'   Training: {X_train.shape[0]} samples')
print(f'   Testing: {X_test.shape[0]} samples')

## Model Training - All Algorithms

### Understanding the Models:

**1. Linear Regression** (Baseline)
- Simplest model: assumes linear relationship
- Fast, interpretable, but limited for complex patterns

**2. Decision Tree**
- Creates a tree of if-else rules
- Can capture non-linear patterns but prone to overfitting

**3. Random Forest** (Bagging Ensemble)
- Combines multiple decision trees ("forest")
- Each tree trains on random subset of data
- Averages predictions to reduce overfitting

**4. Gradient Boosting** (Boosting Ensemble)
- Builds trees sequentially
- Each new tree corrects errors from previous trees
- Very powerful for structured/tabular data

**5. XGBoost** (Advanced Gradient Boosting)
- Optimized version of Gradient Boosting
- Faster training, better regularization
- Industry standard for competitions

**6. LightGBM** (Fast Gradient Boosting)
- Microsoft's gradient boosting framework
- Extremely fast, memory efficient
- Great for large datasets

In [ ]:
print('\n' + '=' * 70)
print('TRAINING ALL MODELS')
print('=' * 70)

# Store all models and metrics
models = {}
predictions = {}
metrics_results = {}

# Define all models
model_configs = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_split=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, min_samples_split=5, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, subsample=0.8, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, num_leaves=31, random_state=42, n_jobs=-1, verbose=-1)
}

# Train each model
for model_name, model in model_configs.items():
    print(f'\n🚀 Training {model_name}...')
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, np.maximum(y_pred, 0.01))
    r2 = r2_score(y_test, y_pred)
    
    # Store results
    models[model_name] = model
    predictions[model_name] = y_pred
    metrics_results[model_name] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R²': r2}
    
    print(f'   ✓ {model_name} trained')
    print(f'     MAE: {mae:.2f} | RMSE: {rmse:.2f} | MAPE: {mape:.2%} | R²: {r2:.4f}')

print('\n✅ All models trained successfully!')

## Comprehensive Model Comparison

In [ ]:
print('\n' + '=' * 80)
print('COMPREHENSIVE MODEL COMPARISON')
print('=' * 80)

# Create comparison DataFrame
comparison_df = pd.DataFrame(metrics_results).T
comparison_df = comparison_df.round(4)

print('\n📊 Performance Metrics (All Models):')
print(comparison_df.to_string())

# Find best model for each metric
print('\n🏆 BEST MODEL BY METRIC:')
print('-' * 80)
print(f"MAE (lower is better):  {comparison_df['MAE'].idxmin()} - {comparison_df['MAE'].min():.2f}")
print(f"RMSE (lower is better): {comparison_df['RMSE'].idxmin()} - {comparison_df['RMSE'].min():.2f}")
print(f"MAPE (lower is better): {comparison_df['MAPE'].idxmin()} - {comparison_df['MAPE'].min():.2%}")
print(f"R² (higher is better):  {comparison_df['R²'].idxmax()} - {comparison_df['R²'].max():.4f}")

# Rank models
print('\n📈 MODEL RANKING (by MAE):')
print('-' * 80)
ranked = comparison_df.sort_values('MAE')
for i, (model_name, row) in enumerate(ranked.iterrows(), 1):
    print(f"{i}. {model_name:20s} - MAE: {row['MAE']:.2f}, RMSE: {row['RMSE']:.2f}, R²: {row['R²']:.4f}")

## Visualization - Model Comparison

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. MAE Comparison
ax1 = axes[0, 0]
comparison_df['MAE'].sort_values().plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Mean Absolute Error (MAE) - Lower is Better', fontsize=14, fontweight='bold')
ax1.set_xlabel('MAE')
ax1.grid(axis='x', alpha=0.3)

# 2. RMSE Comparison
ax2 = axes[0, 1]
comparison_df['RMSE'].sort_values().plot(kind='barh', ax=ax2, color='coral')
ax2.set_title('Root Mean Squared Error (RMSE) - Lower is Better', fontsize=14, fontweight='bold')
ax2.set_xlabel('RMSE')
ax2.grid(axis='x', alpha=0.3)

# 3. MAPE Comparison
ax3 = axes[1, 0]
comparison_df['MAPE'].sort_values().plot(kind='barh', ax=ax3, color='lightgreen')
ax3.set_title('Mean Absolute Percentage Error (MAPE) - Lower is Better', fontsize=14, fontweight='bold')
ax3.set_xlabel('MAPE')
ax3.grid(axis='x', alpha=0.3)

# 4. R² Comparison
ax4 = axes[1, 1]
comparison_df['R²'].sort_values(ascending=False).plot(kind='barh', ax=ax4, color='orchid')
ax4.set_title('R² Score - Higher is Better', fontsize=14, fontweight='bold')
ax4.set_xlabel('R² Score')
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Comparison chart saved: model_comparison_all_metrics.png')

## Select Best Model

In [ ]:
# Select best model based on MAE
best_model_name = comparison_df['MAE'].idxmin()
best_model = models[best_model_name]
best_metrics = metrics_results[best_model_name]

print('\n' + '=' * 80)
print('SELECTED BEST MODEL')
print('=' * 80)
print(f'\n🏆 WINNER: {best_model_name}')
print(f'\n📊 Performance:')
print(f'   MAE:  {best_metrics["MAE"]:.2f}')
print(f'   RMSE: {best_metrics["RMSE"]:.2f}')
print(f'   MAPE: {best_metrics["MAPE"]:.2%}')
print(f'   R²:   {best_metrics["R²"]:.4f}')

print(f'\n📝 Why {best_model_name}?')
print('-' * 80)
if 'XGBoost' in best_model_name or 'LightGBM' in best_model_name or 'Gradient' in best_model_name:
    print('''
Gradient Boosting methods (GB, XGBoost, LightGBM) excel because:
1. ✓ Handle non-linear patterns in sales data
2. ✓ Work well with time-series features (lags, rolling stats)
3. ✓ Robust to outliers and missing data
4. ✓ Provide feature importance for business insights
5. ✓ Industry-proven for forecasting tasks
''')
elif 'Random Forest' in best_model_name:
    print('''
Random Forest excels because:
1. ✓ Ensemble of trees reduces overfitting
2. ✓ Handles non-linear relationships well
3. ✓ Robust and stable predictions
4. ✓ Works well with many features
''')
else:
    print(f'{best_model_name} performed best on the evaluation metrics.')

## Feature Importance (Best Model)

In [ ]:
# Get feature importance if available
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print('\n🚀 Top 10 Important Features:')
    print(feature_importance.head(10).to_string(index=False))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    top_features = feature_importance.head(10)
    ax.barh(top_features['feature'], top_features['importance'], color='steelblue')
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_title(f'Top 10 Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\n✅ Feature importance chart saved!')
else:
    print(f'\n⚠ {best_model_name} does not provide feature importance.')

## Top-Selling Products by Category

In [ ]:
print('\n' + '=' * 80)
print('TOP-SELLING PRODUCTS ANALYSIS')
print('=' * 80)

# Calculate total sales by product and category
product_sales = df.groupby(['category', 'product_name']).agg({
    'quantity': 'sum',
    'total': 'sum'
}).reset_index()

# Top 10 Beverages
print('\n☕ TOP 10 BEVERAGE PRODUCTS:')
print('-' * 80)
beverages = product_sales[product_sales['category'] == 'Beverage'].sort_values('quantity', ascending=False).head(10)
for i, row in enumerate(beverages.itertuples(), 1):
    print(f"{i:2d}. {row.product_name:30s} - {row.quantity:>6.0f} units | ₱{row.total:>10,.2f}")

# Top 10 Pastries
print('\n🥐 TOP 10 PASTRY PRODUCTS:')
print('-' * 80)
pastries = product_sales[product_sales['category'] == 'Pastry'].sort_values('quantity', ascending=False).head(10)
for i, row in enumerate(pastries.itertuples(), 1):
    print(f"{i:2d}. {row.product_name:30s} - {row.quantity:>6.0f} units | ₱{row.total:>10,.2f}")

# Save to CSV
top_products = pd.concat([
    beverages.head(10).assign(rank_in_category=range(1, 11)),
    pastries.head(10).assign(rank_in_category=range(1, 11))
])
top_products.to_csv('top_selling_products_by_category.csv', index=False)
print('\n✅ Saved: top_selling_products_by_category.csv')

## Generate 30-Day Forecast

In [ ]:
print('\n' + '=' * 70)
print('GENERATING 30-DAY FORECAST')
print('=' * 70)

last_date = df_features['date'].max()
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30, freq='D')

print(f'\n📅 Forecast Period: {forecast_dates[0].date()} to {forecast_dates[-1].date()}')

daily_forecasts = []
last_row = df_features.iloc[-1][feature_columns].values.reshape(1, -1)

for forecast_date in forecast_dates:
    forecast_features = last_row.copy()
    forecast_features[0][0] = forecast_date.dayofweek
    forecast_features[0][1] = forecast_date.day
    forecast_features[0][2] = forecast_date.month
    forecast_features[0][3] = forecast_date.year
    forecast_features[0][4] = forecast_date.isocalendar()[1]
    forecast_features[0][5] = 1 if forecast_date.dayofweek in [5, 6] else 0
    
    pred = best_model.predict(forecast_features)[0]
    daily_forecasts.append({
        'date': forecast_date.date(),
        'day_name': forecast_date.strftime('%A'),
        'forecast_quantity': max(0, pred)
    })

forecast_df = pd.DataFrame(daily_forecasts)

print(f'\n📊 Daily Forecast (First 10 days):')
print(forecast_df.head(10).to_string(index=False))

monthly_total = forecast_df['forecast_quantity'].sum()
print(f'\n📈 30-Day Total Forecast: {monthly_total:.2f} units')
print(f'📈 Average Daily Forecast: {monthly_total/30:.2f} units')

## Generate Professional Report

In [ ]:
print('\n' + '=' * 70)
print('GENERATING PROFESSIONAL REPORT')
print('=' * 70)

report_content = f"""{'=' * 80}
BANELO SALES FORECASTING - ENHANCED ML MODEL COMPARISON REPORT
{'=' * 80}

📅 TRAINING INFORMATION
{'-' * 80}
Training Period: {df_features['date'].min().date()} to {df_features['date'].max().date()}
Total Records: {len(df_features)}
Unique Products: {df_features['product_name'].nunique()}
Number of Features: {len(feature_columns)}
Models Compared: {len(models)}

{'=' * 80}
COMPREHENSIVE MODEL COMPARISON
{'=' * 80}

"""

for model_name, metrics in metrics_results.items():
    report_content += f"""
{model_name}:
   MAE (Mean Absolute Error): {metrics['MAE']:.2f}
   RMSE (Root Mean Squared Error): {metrics['RMSE']:.2f}
   MAPE (Mean Absolute Percentage Error): {metrics['MAPE']:.2%}
   R² Score: {metrics['R²']:.4f}
"""

report_content += f"""
{'=' * 80}
BEST MODEL SELECTION
{'=' * 80}

🏆 Selected Model: {best_model_name}

Performance:
   MAE:  {best_metrics['MAE']:.2f}
   RMSE: {best_metrics['RMSE']:.2f}
   MAPE: {best_metrics['MAPE']:.2%}
   R²:   {best_metrics['R²']:.4f}

Justification:
The {best_model_name} model was selected based on achieving the best MAE score
among all {len(models)} models tested. This model demonstrated:
- Superior prediction accuracy
- Robust handling of time-series features
- Ability to capture non-linear sales patterns
- Strong generalization on unseen data

{'=' * 80}
30-DAY FORECAST
{'=' * 80}

Forecast Period: {forecast_dates[0].date()} to {forecast_dates[-1].date()}
Total 30-Day Forecast: {monthly_total:.2f} units
Average Daily Forecast: {monthly_total/30:.2f} units

{'=' * 80}
KEY INSIGHTS
{'=' * 80}

✓ Model Comparison Benefits:
  - Compared {len(models)} different algorithms systematically
  - Justified model selection with quantitative metrics
  - Demonstrated {best_model_name} superiority over baseline Linear Regression
  - Improvement over baseline: {((metrics_results['Linear Regression']['MAE'] - best_metrics['MAE']) / metrics_results['Linear Regression']['MAE'] * 100):.1f}%

✓ Business Insights:
  - Accurate sales forecasting enables better inventory management
  - Category-specific analysis helps optimize product mix
  - 30-day forecast supports financial planning

{'=' * 80}

Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

{'=' * 80}
"""

with open('Banelo_Enhanced_Forecasting_Report.txt', 'w') as f:
    f.write(report_content)

print('\n✅ Report saved: Banelo_Enhanced_Forecasting_Report.txt')
print(report_content)

## Save Models & Outputs

In [ ]:
print('\n' + '=' * 70)
print('SAVING MODELS & OUTPUTS')
print('=' * 70)

# Save best model
joblib.dump(best_model, f'{best_model_name.lower().replace(" ", "_")}_model.pkl')
joblib.dump(feature_columns, 'feature_columns.pkl')
joblib.dump(category_encoder, 'category_encoder.pkl')

# Save all model comparison
comparison_df.to_csv('model_comparison_metrics.csv')

# Save forecasts
forecast_df.to_csv('forecasts_30day.csv', index=False)

print('\n✅ Files saved successfully!')
print('\nFiles ready for download:')
print(f'   1. {best_model_name.lower().replace(" ", "_")}_model.pkl (Best model)')
print('   2. feature_columns.pkl')
print('   3. category_encoder.pkl')
print('   4. model_comparison_metrics.csv')
print('   5. top_selling_products_by_category.csv')
print('   6. forecasts_30day.csv')
print('   7. Banelo_Enhanced_Forecasting_Report.txt')
print('   8. model_comparison_all_metrics.png')
print('   9. feature_importance.png (if available)')

## Summary

In [ ]:
print('\n' + '=' * 70)
print('TRAINING COMPLETE! 🎉')
print('=' * 70)

improvement_pct = ((metrics_results['Linear Regression']['MAE'] - best_metrics['MAE']) / metrics_results['Linear Regression']['MAE'] * 100)

print(f"""
✅ SUMMARY:
   Models Compared: {len(models)}
   Best Model: {best_model_name}
   Best MAE: {best_metrics['MAE']:.2f}
   Baseline MAE: {metrics_results['Linear Regression']['MAE']:.2f}
   Improvement: {improvement_pct:.1f}%
   
   Total features: {len(feature_columns)}
   Training records: {len(df_features)}
   30-day forecast: {monthly_total:.2f} units

🎯 NEXT STEPS:
   1. Download all .pkl files
   2. Download all CSV and TXT reports
   3. Move {best_model_name.lower().replace(" ", "_")}_model.pkl to: ml_models/gradient_boosting_model.pkl
   4. Move other .pkl files to: ml_models/ folder
   5. Run: python integrate_ml_model.py

📚 THESIS DELIVERABLES:
   ✓ Comprehensive model comparison ({len(models)} algorithms)
   ✓ Quantitative justification for model selection
   ✓ Performance metrics (MAE, RMSE, MAPE, R²)
   ✓ Feature importance analysis
   ✓ Top-selling products by category
   ✓ Professional research report
   ✓ Visualization charts
""")